# SpatialPPIv2 — Demo Walkthrough

This notebook walks through the full SpatialPPIv2 pipeline:

1. Download a PDB structure
2. Extract ProtT5 embeddings
3. Build a residue contact graph
4. Run inference with the GATv2 model
5. Embed a set of proteins and visualise them with UMAP
6. Query the embedding API

> **Prerequisites**: `pip install -e ".[dev]"` from the repo root, plus model weights in `checkpoint/`.

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from spatialppiv2.utils.config import get_config
from spatialppiv2.utils.tool import Embed, extractPDB, getConfig
from spatialppiv2.utils.model import getModel
from spatialppiv2.utils.dataset import build_data

plt.rcParams.update({'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False})

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 1. Download example PDB structures

We use TP53 and MDM2 as a canonical positive PPI pair, and TP53 + a random non-interactor as a negative.

In [ ]:
from spatialppiv2.data.download_pdbs import gene_to_pdb_id, download_rcsb

Path('data/demo').mkdir(parents=True, exist_ok=True)

pairs = [
    ('TP53',  '1TUP'),
    ('MDM2',  '1Z1M'),
    ('EGFR',  '1IVO'),
]

for gene, pdb_id in pairs:
    out = Path(f'data/demo/{gene}.pdb')
    if not out.exists():
        ok = download_rcsb(pdb_id, out)
        print(f'{gene}: {"✓" if ok else "✗"}')
    else:
        print(f'{gene}: already downloaded')

## 2. Load model and embedder

In [ ]:
yaml_cfg = getConfig('config/default.yaml')
embedder = Embed('Rostlab/prot_t5_xl_uniref50', DEVICE)
yaml_cfg['basic']['num_features'] = embedder.featureLen

ckpt = 'checkpoint/SpatialPPIv2_ProtT5.ckpt'
model = getModel(yaml_cfg, ckpt=ckpt if Path(ckpt).exists() else None).to(DEVICE)
model.eval()
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

## 3. Extract embeddings and build contact graphs

Each protein becomes a graph where nodes are residues (ProtT5 features) and edges connect Cα atoms within 8 Å.

In [ ]:
def load_protein(pdb_path, chain='first'):
    seq, coords = extractPDB(pdb_path, chain)
    emb = embedder.encode(seq)
    print(f'  {Path(pdb_path).stem}: {len(seq)} residues, embedding shape {emb.shape}')
    return seq, coords, emb

print('Loading proteins...')
seq_tp53, coord_tp53, emb_tp53 = load_protein('data/demo/TP53.pdb')
seq_mdm2, coord_mdm2, emb_mdm2 = load_protein('data/demo/MDM2.pdb')
seq_egfr, coord_egfr, emb_egfr = load_protein('data/demo/EGFR.pdb')

In [ ]:
# Build PPI graphs
data_pos = build_data(
    node_feature=torch.cat([emb_tp53, emb_mdm2]),
    coords=[coord_tp53, coord_mdm2],
    pdb_paths=['data/demo/TP53.pdb', 'data/demo/MDM2.pdb'],
).to(DEVICE)

data_neg = build_data(
    node_feature=torch.cat([emb_tp53, emb_egfr]),
    coords=[coord_tp53, coord_egfr],
    pdb_paths=['data/demo/TP53.pdb', 'data/demo/EGFR.pdb'],
).to(DEVICE)

print('Positive pair graph:', data_pos.data_shape)
print('Negative pair graph:', data_neg.data_shape)

## 4. Run inference

In [ ]:
with torch.no_grad():
    score_pos = model(data_pos).cpu().item()
    score_neg = model(data_neg).cpu().item()

print(f'TP53 × MDM2  (positive):  {score_pos:.4f}')
print(f'TP53 × EGFR  (negative):  {score_neg:.4f}')

fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(['TP53 × MDM2\n(positive)', 'TP53 × EGFR\n(negative)'],
              [score_pos, score_neg],
              color=['#1D9E75', '#E24B4A'], width=0.5)
ax.axhline(0.5, lw=1.5, ls='--', color='#888780', label='Threshold = 0.5')
ax.set_ylabel('Interaction probability')
ax.set_ylim(0, 1)
ax.set_title('SpatialPPIv2 — Single pair inference')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('results/figures/demo_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Protein embedding space (UMAP)

Embed several proteins and visualise their graph-level representations.

In [ ]:
# Gather graph-level embeddings for all three proteins
proteins = {
    'TP53':  (seq_tp53, coord_tp53, emb_tp53, 'data/demo/TP53.pdb'),
    'MDM2':  (seq_mdm2, coord_mdm2, emb_mdm2, 'data/demo/MDM2.pdb'),
    'EGFR':  (seq_egfr, coord_egfr, emb_egfr, 'data/demo/EGFR.pdb'),
}

embeddings = {}
with torch.no_grad():
    for name, (seq, coords, emb, pdb) in proteins.items():
        # Build a pair with itself (we only use h_A from model.embed)
        data = build_data(
            node_feature=torch.cat([emb, emb]),
            coords=[coords, coords],
            pdb_paths=[pdb, pdb],
        ).to(DEVICE)
        h_a, _ = model.embed(data)
        embeddings[name] = h_a.squeeze(0).cpu().numpy()

print('Graph-level embedding dim:', list(embeddings.values())[0].shape)

In [ ]:
# Cosine similarity matrix
names = list(embeddings.keys())
vecs  = np.stack([embeddings[n] for n in names])
sim_matrix = (vecs @ vecs.T)  # L2-normalised, so dot = cosine similarity

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(sim_matrix, vmin=-1, vmax=1, cmap='RdYlGn')
ax.set_xticks(range(len(names)))
ax.set_yticks(range(len(names)))
ax.set_xticklabels(names)
ax.set_yticklabels(names)
plt.colorbar(im, ax=ax, label='Cosine similarity')

for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=10)

ax.set_title('Protein embedding similarity')
plt.tight_layout()
plt.savefig('results/figures/embedding_similarity.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Query the embedding API

Start the API first: `sppi-api --port 8000` in a separate terminal.

In [ ]:
import requests

BASE = 'http://localhost:8000'

# Health check
r = requests.get(f'{BASE}/health')
print('Health:', r.json())

In [ ]:
# Embed via API
r = requests.post(f'{BASE}/embed', json={
    'texts': [seq_tp53[:50], seq_mdm2[:50]],  # truncated for demo
})
body = r.json()
print(f'Embeddings returned: {len(body["embeddings"])}')
print(f'Embedding dim: {len(body["embeddings"][0])}')
print(f'Latency: {body["meta"]["latency_ms"]:.1f} ms')

In [ ]:
# Score via API
r = requests.post(f'{BASE}/score', json={
    'protein_a': str(Path('data/demo/TP53.pdb').resolve()),
    'protein_b': str(Path('data/demo/MDM2.pdb').resolve()),
    'input_type': 'pdb_path',
})
result = r.json()
print(f"TP53 × MDM2 interaction probability: {result['interaction_probability']:.4f}")
print(f"Latency: {result['latency_ms']:.1f} ms")

## 7. Contrastive pre-training demo

Shows how augmentations transform a graph and computes NT-Xent loss for a tiny batch.

In [ ]:
from spatialppiv2.models.contrastive import (
    augment_node_dropout, augment_edge_dropout,
    augment_gaussian_noise, nt_xent_loss, ContrastiveTrainer
)

print('Original graph:', data_pos.data_shape)
print('After node dropout:', augment_node_dropout(data_pos.cpu(), p=0.1).data_shape)
print('After edge dropout:', augment_edge_dropout(data_pos.cpu(), p=0.3).data_shape)

# NT-Xent loss on random embeddings
import torch.nn.functional as F
B, D = 4, 64
z_i = F.normalize(torch.randn(B, D), dim=-1)
z_j = F.normalize(torch.randn(B, D), dim=-1)
loss = nt_xent_loss(z_i, z_j, temperature=0.07)
print(f'NT-Xent loss (random batch of {B}): {loss.item():.4f}')

# Compare with identical views (should be lower)
loss_identical = nt_xent_loss(z_i, z_i.clone(), temperature=0.07)
print(f'NT-Xent loss (identical views): {loss_identical.item():.4f}')